# Notebook 03: Pelatihan Model Deep Learning Multi-Task BiLSTM

**Topik**: Estimasi Tekanan Darah Non-Invasif (SBP, DBP, MAP) dari Sinyal ECG 10 Detik  
**Arsitektur**: Shared BiLSTM Encoder + 3 Dedicated Heads + Adaptive Multi-Task Uncertainty Loss  
**Dataset Sumber**: Berkas `.npz` hasil pra-pemrosesan di folder `data/processed/`

---
### Alur Notebook:
1. **Pemuatan Dataset Preprocessed**: Memuat berkas `.npz` (`part_1_preprocessed.npz`, dst.)
2. **Data Splitting & DataLoader**: Pembagian 80% Train, 10% Validation, 10% Test
3. **Inisialisasi Model & Loss**: Memanggil `ECGMultiTaskBiLSTM` dan `AdaptiveMultiTaskLoss`
4. **Training & Validation Loop**: Proses iterasi pelatihan dengan checkpointing bobot terbaik
5. **Visualisasi Kurva Evaluasi**: Grafik Loss dan MAE (SBP, DBP, MAP) per epoch
6. **Uji Performa pada Test Set**: Scatter Plot Prediksi vs Ground Truth & Distribusi Error

## 1. Import Pustaka & Konfigurasi Lingkungan

In [ ]:
import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

# Konfigurasi root proyek
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.bilstm import ECGMultiTaskBiLSTM
from src.models.loss import AdaptiveMultiTaskLoss
from src.models.dataset import ECGBPDataset

# Deteksi device akselerasi (Apple Silicon MPS / CUDA / CPU)
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device Akselerator Aktif: {device}")

## 2. Pemuatan Dataset Preprocessed (.npz)

In [ ]:
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
npz_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*_preprocessed.npz')))

print(f"Ditemukan {len(npz_files)} berkas .npz hasil pra-pemrosesan:")
for f in npz_files:
    sz = os.path.getsize(f) / (1024 * 1024)
    print(f"  - {os.path.basename(f)} ({sz:.1f} MB)")

# PILIHAN: Memuat 1 part untuk training cepat atau beberapa part
# Anda bisa mengubah npz_files[:1] menjadi npz_files[:3] atau npz_files (semua)
selected_files = npz_files[:1]  # Memuat part_1 sebagai demonstrasi awal
print(f"\nMemuat {len(selected_files)} file terpilih...")
dataset = ECGBPDataset.from_npz_files(selected_files)
print(f"Total epoch siap latih: {len(dataset):,} sampel (10 detik @ 125 Hz)")

## 3. Pembagian Dataset (Train, Validation, Test)

In [ ]:
BATCH_SIZE = 64
SEED = 42

total_len = len(dataset)
train_len = int(0.80 * total_len)
val_len = int(0.10 * total_len)
test_len = total_len - train_len - val_len

train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data Split:")
print(f" - Train Set : {len(train_set):,} epoch ({len(train_loader)} batches)")
print(f" - Val Set   : {len(val_set):,} epoch ({len(val_loader)} batches)")
print(f" - Test Set  : {len(test_set):,} epoch ({len(test_loader)} batches)")

## 4. Inisialisasi Model Multi-Task BiLSTM & Adaptive Loss

In [ ]:
model = ECGMultiTaskBiLSTM(
    input_size=1,
    hidden_size=256,
    num_layers=2,
    lstm_dropout=0.3,
    head_dropout=0.2,
    pooling='last'
).to(device)

criterion = AdaptiveMultiTaskLoss(loss_fn_type='mae', init_log_var=0.0).to(device)

LEARNING_RATE = 1e-3
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-5
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model berhasil diinisialisasi!")
print(f"Total trainable parameters: {num_params:,}")

## 5. Proses Pelatihan (Training Loop)

In [ ]:
NUM_EPOCHS = 10
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

history = {
    'train_loss': [], 'val_loss': [],
    'val_mae_sbp': [], 'val_mae_dbp': [], 'val_mae_map': []
}
best_val_mae = float('inf')

print("Memulai pelatihan...")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    
    # --- TRAINING PHASE ---
    model.train()
    train_loss, n_train_batches = 0.0, 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        sbp_p, dbp_p, map_p = model(x_b)
        loss, _ = criterion(sbp_p, dbp_p, map_p, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item()
        n_train_batches += 1
        
    avg_train_loss = train_loss / max(1, n_train_batches)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, sbp_mae, dbp_mae, map_mae, n_val_batches = 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            sbp_p, dbp_p, map_p = model(x_b)
            loss, m = criterion(sbp_p, dbp_p, map_p, y_b)
            val_loss += loss.item()
            sbp_mae += m['mae_sbp']
            dbp_mae += m['mae_dbp']
            map_mae += m['mae_map']
            n_val_batches += 1
            
    avg_val_loss = val_loss / max(1, n_val_batches)
    avg_sbp_mae = sbp_mae / max(1, n_val_batches)
    avg_dbp_mae = dbp_mae / max(1, n_val_batches)
    avg_map_mae = map_mae / max(1, n_val_batches)
    avg_val_mae = (avg_sbp_mae + avg_dbp_mae + avg_map_mae) / 3.0
    
    scheduler.step(avg_val_loss)
    elapsed = time.time() - t0
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_mae_sbp'].append(avg_sbp_mae)
    history['val_mae_dbp'].append(avg_dbp_mae)
    history['val_mae_map'].append(avg_map_mae)
    
    is_best = avg_val_mae < best_val_mae
    if is_best:
        best_val_mae = avg_val_mae
        torch.save(model.state_dict(), best_model_path)
        mark = "★ BEST"
    else:
        mark = ""
        
    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {avg_train_loss:.3f} | Val Loss: {avg_val_loss:.3f} | Val MAE -> SBP: {avg_sbp_mae:.2f}, DBP: {avg_dbp_mae:.2f}, MAP: {avg_map_mae:.2f} mmHg {mark}")

print(f"\nPelatihan selesai dalam {(time.time() - start_time)/60:.1f} menit!")
print(f"Bobot model terbaik tersimpan di: {best_model_path}")

## 6. Visualisasi Kurva Konvergensi (Loss & MAE)

In [ ]:
epochs_arr = np.arange(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))

# Subplot 1: Total Loss
ax1.plot(epochs_arr, history['train_loss'], 'o-', label='Train Loss', color='#1f77b4', lw=2)
ax1.plot(epochs_arr, history['val_loss'], 's--', label='Val Loss', color='#d62728', lw=2)
ax1.set_title("Kurva Konvergensi Adaptive Multi-Task Loss", fontsize=11, fontweight='bold')
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Total Loss")
ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.5)

# Subplot 2: MAE per Task
ax2.plot(epochs_arr, history['val_mae_sbp'], 'o-', label='Val SBP MAE', color='#1f77b4', lw=2)
ax2.plot(epochs_arr, history['val_mae_dbp'], 's-', label='Val DBP MAE', color='#2ca02c', lw=2)
ax2.plot(epochs_arr, history['val_mae_map'], '^-', label='Val MAP MAE', color='#ff7f0e', lw=2)
ax2.set_title("Mean Absolute Error (MAE) per Task pada Validation Set", fontsize=11, fontweight='bold')
ax2.set_xlabel("Epoch"); ax2.set_ylabel("MAE (mmHg)")
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Evaluasi Akhir pada Test Set (Data Uji Independen)

In [ ]:
# Muat bobot model terbaik
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds = []
all_trues = []

with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b = x_b.to(device)
        preds = model.predict_all(x_b)
        all_preds.append(preds.cpu().numpy())
        all_trues.append(y_b.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_trues = np.concatenate(all_trues, axis=0)

sbp_err = np.abs(all_preds[:, 0] - all_trues[:, 0])
dbp_err = np.abs(all_preds[:, 1] - all_trues[:, 1])
map_err = np.abs(all_preds[:, 2] - all_trues[:, 2])

print("=" * 60)
print("  HASIL EVALUASI MODEL PADA TEST SET:")
print(f"  - SBP MAE : {np.mean(sbp_err):.2f} +/- {np.std(sbp_err):.2f} mmHg")
print(f"  - DBP MAE : {np.mean(dbp_err):.2f} +/- {np.std(dbp_err):.2f} mmHg")
print(f"  - MAP MAE : {np.mean(map_err):.2f} +/- {np.std(map_err):.2f} mmHg")
print("=" * 60)

# Scatter plot Prediksi vs Ground Truth
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
targets_names = ['SBP', 'DBP', 'MAP']
colors = ['#1f77b4', '#2ca02c', '#ff7f0e']

for i in range(3):
    ax = axes[i]
    y_t = all_trues[:, i]
    y_p = all_preds[:, i]
    ax.scatter(y_t, y_p, alpha=0.3, color=colors[i], s=12)
    min_val = min(np.min(y_t), np.min(y_p)) - 5
    max_val = max(np.max(y_t), np.max(y_p)) + 5
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5, label='Garis Ideal (y=x)')
    ax.set_title(f"Korelasi {targets_names[i]}: Ground Truth vs Prediksi", fontsize=11, fontweight='bold')
    ax.set_xlabel(f"{targets_names[i]} Ground Truth (mmHg)")
    ax.set_ylabel(f"{targets_names[i]} Prediksi Model (mmHg)")
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Evaluasi Prediksi Model Multi-Task BiLSTM pada Data Uji", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()